# vLLM Evaluation
This notebook handles the vLLM evaluation step.

In [1]:
# Cell 1b: vLLM Install
# Do NOT pin torch — Kaggle's system torch 2.11 is what all other packages expect.
# vLLM 0.6.3 is too old for Qwen2.5-VL's M-RoPE. We need 0.7.2+ for proper support.
!pip install -q vllm qwen-vl-utils kaggle
!pip install -q nvidia-cuda-nvrtc-cu12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.1/250.1 MB 7.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 58.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.0/458.0 MB 3.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 83.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.9/184.9 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 83.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.7 MB/s eta 0:00:00
   ━━━━━

## Step C: Run vLLM Evaluation with Fallbacks
Intelligently falls back for both the model and the evaluation script if they aren't already downloaded.

In [2]:
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# import os

# try:
#     # Fetch the token securely from Kaggle Secrets
#     user_secrets = UserSecretsClient()
#     hf_token = user_secrets.get_secret("HF_TOKEN")
    
#     # Log in to Hugging Face Hub
#     login(token=hf_token)
    
#     # Also set it as an environment variable (vLLM automatically looks for this)
#     os.environ["HF_TOKEN"] = hf_token 
    
#     print("✅ Successfully logged into Hugging Face!")
# except Exception as e:
#     print("❌ Could not find HF_TOKEN in Kaggle Secrets. Please add it via Add-ons -> Secrets.")
#     print("Error:", e)

In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("✅ Successfully logged into Hugging Face!")
except Exception as e:
    print("❌ Could not find HF_TOKEN in Kaggle Secrets.")
    print("Error:", e)

import json
import shutil

model_path = "/tmp/disasterm3_merged"
if not os.path.exists(model_path):
    print("Local merged model not found in /tmp. Falling back to Hugging Face Hub...")
    model_path = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"
else:
    print(f"Using local merged model at {model_path}")

repo_path = "/tmp/DisasterM3_Repo"
if os.path.exists(repo_path):
    shutil.rmtree(repo_path)
!git clone https://github.com/Junjue-Wang/DisasterM3.git {repo_path}
print("✓ Fresh clone of DisasterM3 repo")

# ── Patch run_vllm.py ──
vllm_script = f"{repo_path}/pyscripts/run_vllm.py"
with open(vllm_script, "r") as f:
    lines = f.readlines()

new_lines = []
skip_until_llm = False
for line in lines:
    # 1. Fix the LLM initialization (T4x2 Tensor Parallelism)
    if "EngineArgs(" in line and "engine_args" in line:
        skip_until_llm = True
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f"{indent}vlm_model = LLM(\n")
        new_lines.append(f"{indent}    model=args.model_id,\n")
        new_lines.append(f"{indent}    limit_mm_per_prompt={{'image': 2}},\n")
        new_lines.append(f"{indent}    gpu_memory_utilization=0.92,\n")
        new_lines.append(f"{indent}    max_model_len=4096,\n")
        new_lines.append(f"{indent}    enforce_eager=True,\n")
        new_lines.append(f"{indent}    trust_remote_code=True,\n")
        new_lines.append(f"{indent}    tensor_parallel_size=2,\n")
        new_lines.append(f"{indent})\n")
        continue
    if skip_until_llm:
        if "LLM(**engine_args)" in line:
            skip_until_llm = False
            continue
        else:
            continue
            
    # 2. Fix the SamplingParams setattr crash
    if "setattr(sampling_params" in line:
        indent = line[:len(line) - len(line.lstrip())]
        new_lines.append(f"{indent}pass  # [PATCHED] Bypass read-only setattr crash\n")
        continue

    # 3. Fix the original authors' relational reasoning bug (Missing "images" folder)
    if 'image_path = join(f"{PROJECT_ROOT}/data", data_dict["image_path"]' in line:
        line = line.replace('f"{PROJECT_ROOT}/data", data_dict', 'f"{PROJECT_ROOT}/data", "images", data_dict')
        
    # 4. Fix the original authors' relational reasoning bug (options_str typo)
    if 'options_str=data_dict["option_str"])' in line:
        line = line.replace('option_str', 'options_str')

    # 5. [NEW FIX] Globally fix Windows backslashes in all image paths at runtime
    line = line.replace('data_dict["pre_image_path"]', 'data_dict["pre_image_path"].replace("\\\\", "/")')
    line = line.replace('data_dict["post_image_path"]', 'data_dict["post_image_path"].replace("\\\\", "/")')
    line = line.replace('data_dict["image_path"]', 'data_dict["image_path"].replace("\\\\", "/")')

    new_lines.append(line)

with open(vllm_script, "w") as f:
    f.writelines(new_lines)
print("✓ Patched run_vllm.py (FP16 + SamplingParams + Relational Reasoning + Path fixes)")


# ── Data Restructuring ──
bench_path = "/kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2"
data_dir = "/tmp/data"
os.makedirs(f"{data_dir}/images/test_images", exist_ok=True)
os.makedirs(f"{data_dir}/images/masks", exist_ok=True)

!cp -rsn {bench_path}/test_images/* {data_dir}/images/test_images/ 2>/dev/null || true
!cp -rsn {bench_path}/masks/* {data_dir}/images/masks/ 2>/dev/null || true

bench_json_path = f"{bench_path}/benchmark_release.json"
print(f"Splitting {bench_json_path} into evaluation subsets...")
with open(bench_json_path, 'r', encoding='utf-8') as f:
    raw_json = f.read()

# [NEW FIX] Replace all Windows backslashes with Linux forward slashes!
raw_json = raw_json.replace("\\\\", "/")
bench_data = json.loads(raw_json)

task_map = {
    "Disaster Bearing Bodies Recognition": "bearing_body",
    "Building Damage Counting": "building_damage_counting",
    "Disaster Type Recognition": "disaster_type",
    "Road Damage Counting": "road_damage_counting",
    "Disaster Scene Recognition": "landuse",
    "Relational Reasoning": "relational_reasoning_qa"
}

subsets_data = {v: [] for v in task_map.values()}
for entry in bench_data:
    task = entry.get("task")
    if task in task_map:
        subsets_data[task_map[task]].append(entry)

for subset_name, data in subsets_data.items():
    with open(f"{data_dir}/{subset_name}.json", 'w', encoding='utf-8') as f:
        json.dump(data, f)

# ── Run Evaluation ──
# --- SPLIT 1: Run these 3 first (~6-7 hours) ---
subsets = ["bearing_body", "building_damage_counting", "disaster_type"]

# --- SPLIT 2: Next session, comment out Split 1 and uncomment Split 2 ---
# subsets = ["road_damage_counting", "landuse", "relational_reasoning_qa"]
hf_token_val = os.environ.get("HF_TOKEN", "")
safe_model_name = model_path.replace("/", "--")

for subset in subsets:
    print(f"\n{'='*50}\nEvaluating {subset}...\n{'='*50}")

    save_dir = f"/tmp/results/{subset}/{safe_model_name}"
    os.makedirs(save_dir, exist_ok=True)
    with open(f"{save_dir}/finished.jsonl", "a") as f:
        pass

    !HF_TOKEN={hf_token_val} PYTHONPATH={repo_path} python {repo_path}/pyscripts/run_vllm.py --model_id {model_path} --subset {subset}


✅ Successfully logged into Hugging Face!
Local merged model not found in /tmp. Falling back to Hugging Face Hub...
Cloning into '/tmp/DisasterM3_Repo'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 82 (delta 24), reused 24 (delta 24), pack-reused 56 (from 1)
Receiving objects: 100% (82/82), 22.22 KiB | 2.78 MiB/s, done.
Resolving deltas: 100% (26/26), done.
✓ Fresh clone of DisasterM3 repo
✓ Patched run_vllm.py (FP16 + SamplingParams + Relational Reasoning + Path fixes)
Splitting /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2/benchmark_release.json into evaluation subsets...

Evaluating bearing_body...
Start from scratch
Reading data from /tmp/data/bearing_body.json
preprocessor_config.json: 100%|████████████████| 575/575 [00:00<00:00, 2.69MB/s]
chat_template.json: 1.05kB [00:00, 437kB/s]
config.json: 1.21kB [00:00, 3.08MB/s]
tokenizer_config.json: 5.78kB [00:00, 14.9MB/

In [4]:
# ── Save Results (Working Directory + Kaggle Dataset) ──
import os
import json
import shutil
from kaggle_secrets import UserSecretsClient

source_dir = "/tmp/results"

# =================================================================
# 1. Zip to Kaggle Working Directory
# =================================================================
dest_zip = "/kaggle/working/disasterm3_eval_results"
print(f"Compressing {source_dir} to {dest_zip}.zip ...")
shutil.make_archive(dest_zip, 'zip', source_dir)

size_mb = os.path.getsize(f"{dest_zip}.zip") / (1024 * 1024)
print(f"✓ Results successfully archived! Size: {size_mb:.2f} MB")

# =================================================================
# 2. Upload directly to Kaggle Datasets
# =================================================================
# Authenticate with Kaggle API
try:
    user_secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
except Exception as e:
    print("\n❌ Could not find KAGGLE_USERNAME or KAGGLE_KEY in Kaggle Secrets! Please add them to push to datasets.")
    raise e

# Prepare the dataset metadata
dataset_slug = "disasterm3-evaluation-results"  # Lowercase alphanumeric and hyphens only!
username = os.environ["KAGGLE_USERNAME"]

metadata = {
  "title": "DisasterM3 Evaluation Results",
  "id": f"{username}/{dataset_slug}",
  "licenses": [{"name": "CC0-1.0"}]
}

with open(f"{source_dir}/dataset-metadata.json", "w") as f:
    json.dump(metadata, f)

print(f"\nUploading {source_dir} to Kaggle Datasets as '{dataset_slug}'...")

# Attempt to create the dataset. If it already exists, push a new version instead!
!kaggle datasets create -p {source_dir} --dir-mode zip 2>/dev/null || kaggle datasets version -p {source_dir} -m "Evaluation Update" --dir-mode zip

print("✓ Upload complete! Your results are safely preserved in both /kaggle/working and Kaggle Datasets.")


Compressing /tmp/results to /kaggle/working/disasterm3_eval_results.zip ...
✓ Results successfully archived! Size: 0.01 MB

Uploading /tmp/results to Kaggle Datasets as 'disasterm3-evaluation-results'...
Starting upload for file disaster_type.zip
Upload successful: disaster_type.zip (4KB)
Starting upload for file bearing_body.zip
Upload successful: bearing_body.zip (5KB)
Starting upload for file building_damage_counting.zip
Upload successful: building_damage_counting.zip (4KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/abrarmohammedtanzim/disasterm3-evaluation-results
✓ Upload complete! Your results are safely preserved in both /kaggle/working and Kaggle Datasets.


In [5]:
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# import os

# try:
#     user_secrets = UserSecretsClient()
#     hf_token = user_secrets.get_secret("HF_TOKEN")
#     login(token=hf_token)
#     os.environ["HF_TOKEN"] = hf_token
#     print("✅ Successfully logged into Hugging Face!")
# except Exception as e:
#     print("❌ Could not find HF_TOKEN in Kaggle Secrets.")
#     print("Error:", e)

# import json
# import shutil

# model_path = "/tmp/disasterm3_merged"
# if not os.path.exists(model_path):
#     print("Local merged model not found in /tmp. Falling back to Hugging Face Hub...")
#     model_path = "AbrarAlam/disasterm3-qwen2.5vl7b-mergedFP"
# else:
#     print(f"Using local merged model at {model_path}")

# repo_path = "/tmp/DisasterM3_Repo"
# if os.path.exists(repo_path):
#     shutil.rmtree(repo_path)
# !git clone https://github.com/Junjue-Wang/DisasterM3.git {repo_path}
# print("✓ Fresh clone of DisasterM3 repo")

# # ── Patch run_vllm.py ──
# vllm_script = f"{repo_path}/pyscripts/run_vllm.py"
# with open(vllm_script, "r") as f:
#     lines = f.readlines()

# new_lines = []
# skip_until_llm = False
# for line in lines:
#     # 1. Fix the LLM initialization (T4x2 Tensor Parallelism)
#     if "EngineArgs(" in line and "engine_args" in line:
#         skip_until_llm = True
#         indent = line[:len(line) - len(line.lstrip())]
#         new_lines.append(f"{indent}vlm_model = LLM(\n")
#         new_lines.append(f"{indent}    model=args.model_id,\n")
#         new_lines.append(f"{indent}    limit_mm_per_prompt={{'image': 2}},\n")
#         new_lines.append(f"{indent}    gpu_memory_utilization=0.92,\n")
#         new_lines.append(f"{indent}    max_model_len=4096,\n")
#         new_lines.append(f"{indent}    enforce_eager=True,\n")
#         new_lines.append(f"{indent}    trust_remote_code=True,\n")
#         new_lines.append(f"{indent}    tensor_parallel_size=2,\n")
#         new_lines.append(f"{indent})\n")
#         continue
#     if skip_until_llm:
#         if "LLM(**engine_args)" in line:
#             skip_until_llm = False
#             continue
#         else:
#             continue
            
#     # 2. Fix the SamplingParams setattr crash
#     if "setattr(sampling_params" in line:
#         indent = line[:len(line) - len(line.lstrip())]
#         new_lines.append(f"{indent}pass  # [PATCHED] Bypass read-only setattr crash\n")
#         continue

#     # 3. Fix the original authors' relational reasoning bug (Missing "images" folder)
#     if 'image_path = join(f"{PROJECT_ROOT}/data", data_dict["image_path"]' in line:
#         line = line.replace('f"{PROJECT_ROOT}/data", data_dict', 'f"{PROJECT_ROOT}/data", "images", data_dict')
        
#     # 4. Fix the original authors' relational reasoning bug (options_str typo)
#     if 'options_str=data_dict["option_str"])' in line:
#         line = line.replace('option_str', 'options_str')

#     # 5. [NEW FIX] Globally fix Windows backslashes in all image paths at runtime
#     line = line.replace('data_dict["pre_image_path"]', 'data_dict["pre_image_path"].replace("\\\\", "/")')
#     line = line.replace('data_dict["post_image_path"]', 'data_dict["post_image_path"].replace("\\\\", "/")')
#     line = line.replace('data_dict["image_path"]', 'data_dict["image_path"].replace("\\\\", "/")')

#     new_lines.append(line)

# with open(vllm_script, "w") as f:
#     f.writelines(new_lines)
# print("✓ Patched run_vllm.py (FP16 + SamplingParams + Relational Reasoning + Path fixes)")


# # ── Data Restructuring ──
# bench_path = "/kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2"
# data_dir = "/tmp/data"
# os.makedirs(f"{data_dir}/images/test_images", exist_ok=True)
# os.makedirs(f"{data_dir}/images/masks", exist_ok=True)

# !cp -rsn {bench_path}/test_images/* {data_dir}/images/test_images/ 2>/dev/null || true
# !cp -rsn {bench_path}/masks/* {data_dir}/images/masks/ 2>/dev/null || true

# bench_json_path = f"{bench_path}/benchmark_release.json"
# print(f"Splitting {bench_json_path} into evaluation subsets...")
# with open(bench_json_path, 'r', encoding='utf-8') as f:
#     raw_json = f.read()

# # [NEW FIX] Replace all Windows backslashes with Linux forward slashes!
# raw_json = raw_json.replace("\\\\", "/")
# bench_data = json.loads(raw_json)

# task_map = {
#     "Disaster Bearing Bodies Recognition": "bearing_body",
#     "Building Damage Counting": "building_damage_counting",
#     "Disaster Type Recognition": "disaster_type",
#     "Road Damage Counting": "road_damage_counting",
#     "Disaster Scene Recognition": "landuse",
#     "Relational Reasoning": "relational_reasoning_qa"
# }

# subsets_data = {v: [] for v in task_map.values()}
# for entry in bench_data:
#     task = entry.get("task")
#     if task in task_map:
#         subsets_data[task_map[task]].append(entry)

# for subset_name, data in subsets_data.items():
#     with open(f"{data_dir}/{subset_name}.json", 'w', encoding='utf-8') as f:
#         json.dump(data, f)

# # ── Run Evaluation ──
# # --- SPLIT 1: Run these 3 first (~6-7 hours) ---
# # subsets = ["bearing_body", "building_damage_counting", "disaster_type"]

# # --- SPLIT 2: Next session, comment out Split 1 and uncomment Split 2 ---
# subsets = ["road_damage_counting", "landuse", "relational_reasoning_qa"]
# hf_token_val = os.environ.get("HF_TOKEN", "")
# safe_model_name = model_path.replace("/", "--")

# for subset in subsets:
#     print(f"\n{'='*50}\nEvaluating {subset}...\n{'='*50}")

#     save_dir = f"/tmp/results/{subset}/{safe_model_name}"
#     os.makedirs(save_dir, exist_ok=True)
#     with open(f"{save_dir}/finished.jsonl", "a") as f:
#         pass

#     !HF_TOKEN={hf_token_val} PYTHONPATH={repo_path} python {repo_path}/pyscripts/run_vllm.py --model_id {model_path} --subset {subset}


In [6]:
# # ── Save Results (Working Directory + Kaggle Dataset) ──
# import os
# import json
# import shutil
# from kaggle_secrets import UserSecretsClient

# source_dir = "/tmp/results"

# # =================================================================
# # 1. Zip to Kaggle Working Directory
# # =================================================================
# dest_zip = "/kaggle/working/disasterm3_eval_results"
# print(f"Compressing {source_dir} to {dest_zip}.zip ...")
# shutil.make_archive(dest_zip, 'zip', source_dir)

# size_mb = os.path.getsize(f"{dest_zip}.zip") / (1024 * 1024)
# print(f"✓ Results successfully archived! Size: {size_mb:.2f} MB")

# # =================================================================
# # 2. Upload directly to Kaggle Datasets
# # =================================================================
# # Authenticate with Kaggle API
# try:
#     user_secrets = UserSecretsClient()
#     os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
#     os.environ["KAGGLE_KEY"] = user_secrets.get_secret("KAGGLE_KEY")
# except Exception as e:
#     print("\n❌ Could not find KAGGLE_USERNAME or KAGGLE_KEY in Kaggle Secrets! Please add them to push to datasets.")
#     raise e

# # Prepare the dataset metadata
# dataset_slug = "disasterm3-evaluation-results"  # Lowercase alphanumeric and hyphens only!
# username = os.environ["KAGGLE_USERNAME"]

# metadata = {
#   "title": "DisasterM3 Evaluation Results",
#   "id": f"{username}/{dataset_slug}",
#   "licenses": [{"name": "CC0-1.0"}]
# }

# with open(f"{source_dir}/dataset-metadata.json", "w") as f:
#     json.dump(metadata, f)

# print(f"\nUploading {source_dir} to Kaggle Datasets as '{dataset_slug}'...")

# # Attempt to create the dataset. If it already exists, push a new version instead!
# !kaggle datasets create -p {source_dir} --dir-mode zip 2>/dev/null || kaggle datasets version -p {source_dir} -m "Evaluation Update" --dir-mode zip

# print("✓ Upload complete! Your results are safely preserved in both /kaggle/working and Kaggle Datasets.")
